In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: c:\QuantBricker
Fixed Income Library is loaded.


## Build multi-indices Yield Curve sub model

### Build method collection for YC

In [2]:
bm_list_yc = []

# SOFR index curve
content_sofr = {
    "TARGET" : "SOFR-1B",
    "OVERNIGHT INDEX FUTURE" : "SOFR-FUTURE-3M",
    "OVERNIGHT INDEX SWAP" : "USD-SOFR-OIS",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_INDEX", content_sofr))

# SOFR funding curve
content_sofr_funding = {
    "TARGET": "SOFR-1B-FLAT",
    "SPREAD ZERO RATE": "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_FUNDING", content_sofr_funding))

# FF index curve, referenced to SOFR
content_ff = {
    "TARGET": "FF-1B",
    "REFERENCE INDEX": "SOFR-1B",
    "OVERNIGHT INDEX BASIS SWAP": "USD-FF-3M-OVER-USD-SOFR-OIS-3M",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_INDEX", content_ff))

# FF funding curve
content_ff_funding = {
    "TARGET": "FF-1B-FLAT",
    "SPREAD ZERO RATE": "FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_FUNDING", content_ff_funding))

# Common YC build method
content_common = {
    "TARGET": "USD",
    "FUNDING PARAMETERS": "USD-FUNDING-PARAMETERS",
    "SOLVER": "BRENTQ",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_COMMON", content_common))

build_method_collection_yc = qfCreateModelBuildMethodCollection(bm_list_yc)
build_method_collection_yc.display()

,Name,Value
0,YIELD_CURVE_INDEX,SOFR-1B
1,YIELD_CURVE_FUNDING,SOFR-1B-FLAT
2,YIELD_CURVE_INDEX,FF-1B
3,YIELD_CURVE_FUNDING,FF-1B-FLAT
4,YIELD_CURVE_COMMON,USD


### Create market data collection

In [3]:
### ois futures
df_fut = pd.DataFrame(
    [
        ["2026-03-19x2026-06-18", 96.44],
        ["2026-06-18x2026-09-17", 96.70],
        ["2026-09-17x2026-12-10", 96.85],
        ["2026-12-10x2027-03-17", 96.90],
        ["2027-03-17x2027-06-16", 96.91],
        ["2027-06-16x2027-09-15", 96.89],
        ["2027-09-15x2027-12-15", 96.85],
        ["2027-12-15x2028-03-15", 96.81],
        ["2028-03-15x2028-06-21", 96.76],
        ["2028-06-21x2028-09-20", 96.71],
        ["2028-09-20x2028-12-20", 96.66],
        ["2028-12-20x2029-03-21", 96.61],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_fut = qfCreateData1D("OVERNIGHT INDEX FUTURE", "SOFR-FUTURE-3M", df_fut)

In [4]:
### ois swap
df_swap = pd.DataFrame(
    [
        ["4Y", 0.03358],
        ["5Y", 0.03422],
        ["6Y", 0.03491],
        ["7Y", 0.03560],
        ["8Y", 0.03624],
        ["9Y", 0.03685],
        ["10Y", 0.03742],
        ["12Y", 0.03849],
        ["15Y", 0.03974],
        ["20Y", 0.04087],
        ["25Y", 0.04110],
        ["30Y", 0.04089],
        ["35Y", 0.04044],
        ["40Y", 0.03996],
        ["50Y", 0.03887],
        ["60Y", 0.03772],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_swap = qfCreateData1D("OVERNIGHT INDEX SWAP", "USD-SOFR-OIS", df_swap)

In [5]:
### ois basis swap
df_basis_swap = pd.DataFrame([    
    ['1Y', 0.0005],
    ['2Y', 0.0003],
    ['3Y', 0.0003],
    ['4Y', 0.0002],
    ['5Y', 0.0001]],
    columns=['axis1', 'values']).set_index('axis1')
data_basis_swap = qfCreateData1D('OVERNIGHT INDEX BASIS SWAP', 'USD-FF-3M-OVER-USD-SOFR-OIS-3M', df_basis_swap)

In [6]:
### spread zero
df_spread_zero_rate_rfr = pd.DataFrame(
    [['1Y', 0.],
     ['5Y', 0.],
     ['10Y', 0.],
     ['20Y', 0.],
     ['30Y', 0.]], 
    columns=['axis1', 'values']).set_index('axis1')
data_szr_rfr = qfCreateData1D('SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD', df_spread_zero_rate_rfr)

df_spread_zero_rate_ff = pd.DataFrame(
    [['1Y', 0.0002],
     ['5Y', 0.0003],
     ['10Y', 0.0003],
     ['20Y', 0.0003],
     ['30Y', 0.0004]], 
    columns=['axis1', 'values']).set_index('axis1')
data_szr_ff = qfCreateData1D('SPREAD ZERO RATE', 'FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD', df_spread_zero_rate_ff)

In [7]:
### funding parameter table
df_dg = pd.DataFrame([
    ['Overnight Index Future', 'SOFR-FUTURE-3M', 'SOFR-1B-FLAT'],
    ['Overnight Index Swap', 'USD-SOFR-OIS', 'SOFR-1B-FLAT'],
    ['Overnight Index Basis Swap', 'USD-FF-3M-OVER-USD-SOFR-OIS-3M', 'SOFR-1B-FLAT']], 
    columns=['DATA TYPE', 'DATA CONVENTION', 'FUNDING IDENTIFIER'])
data_fpt = qfCreateDataGeneric('DATA GENERIC', 'USD-FUNDING-PARAMETERS', df_dg)

In [8]:
### pack up all data into data collection
data_collection_yc = qfCreateDataCollection([data_fut, data_swap, data_basis_swap, data_szr_rfr, data_szr_ff, data_fpt])
data_collection_yc.display()

,Data Shape,Data Type,Data Convention
0,DATA1D,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M
1,DATA1D,OVERNIGHT INDEX SWAP,USD-SOFR-OIS
2,DATA1D,OVERNIGHT INDEX BASIS SWAP,USD-FF-3M-OVER-USD-SOFR-OIS-3M
3,DATA1D,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD
4,DATA1D,SPREAD ZERO RATE,FF-1B-FLAT-OVER-FF-1B-ZERO-SPREAD
5,DATAGENERIC,DATA GENERIC,USD-FUNDING-PARAMETERS


### Create Model Yield Curve

In [9]:
value_date = "2026-02-11"
yc_model = qfCreateModel(value_date, "YIELD_CURVE", data_collection_yc, build_method_collection_yc)
path = "serialized/yc_model_calibrated.pickle"
qfWriteModelObjectToFile(yc_model, path)

'DONE'

In [10]:
yc_model.components_.keys()

dict_keys(['SOFR-1B-FLAT', 'FF-1B-FLAT', 'SOFRON Actual/360', 'FedFundsON Actual/360'])

In [11]:
# Quick check: discount factors under SOFR funding vs FF funding
expiry_date = "2027-06-17"
expiry_date_adj = qfMoveToBusinessDay(expiry_date, "F", "USGS")

df_sofr = qfDiscountFactor(yc_model, "SOFR-1B", expiry_date_adj)
df_ff = qfDiscountFactor(yc_model, "FF-1B", expiry_date_adj)

print(f"Discount factor up to {expiry_date_adj} under SOFR index curve: {df_sofr:.10f}")
print(f"Discount factor up to {expiry_date_adj} under FF index curve  : {df_ff:.10f}")

Discount factor up to 2027-06-17 under SOFR index curve: 0.9566501447
Discount factor up to 2027-06-17 under FF index curve  : 0.9559866471


## Build SABR Model

### Create SABR build method collection

In [12]:
bm_list_sabr = []

# Swaption SABR component
content_swaption = {
    "TARGET": "SOFR-1B-SWAPTION",
    "SWAPTION NORMAL VOLATILITY": "USD-SOFR-SWAPTION",
    "SWAPTION SABR BETA": "USD-SOFR-SWAPTION",
    "SWAPTION SABR NU": "USD-SOFR-SWAPTION",
    "SWAPTION SABR RHO": "USD-SOFR-SWAPTION",
    "VOL INTERPOLATION DOMAIN": "NORMAL VOLATILITY",
    "INTERPOLATION METHOD": "LINEAR",
    "EXTRAPOLATION METHOD": "FLAT",
    "BUSINESSDAY CONVENTION": "F",
    "HOLIDAY CONVENTION": "USGS",
    "SHIFT": 0.04,
}
bm_list_sabr.append(qfCreateBuildMethod("IR_SABR", content_swaption))

# Capfloor SABR component
content_capfloor = {
    "TARGET": "SOFR-1B-CAPFLOOR",
    "SWAPTION NORMAL VOLATILITY": "USD-SOFR-CAPFLOOR",
    "SWAPTION SABR BETA": "USD-SOFR-CAPFLOOR",
    "SWAPTION SABR NU": "USD-SOFR-CAPFLOOR",
    "SWAPTION SABR RHO": "USD-SOFR-CAPFLOOR",
    "VOL INTERPOLATION DOMAIN": "NORMAL VOLATILITY",
    "INTERPOLATION METHOD": "LINEAR",
    "EXTRAPOLATION METHOD": "FLAT",
    "BUSINESSDAY CONVENTION": "F",
    "HOLIDAY CONVENTION": "USGS",
    "SHIFT": 0.04,
}
bm_list_sabr.append(qfCreateBuildMethod("IR_SABR", content_capfloor))

bm_collection_sabr = qfCreateModelBuildMethodCollection(bm_list_sabr)
bm_collection_sabr.display()

,Name,Value
0,IR_SABR,SOFR-1B-SWAPTION
1,IR_SABR,SOFR-1B-CAPFLOOR


### Create SABR market data

In [13]:
### swaption
swaption_expiries = ['3M', '6M', '1Y', '5Y', '10Y']
swaption_tenors = ['1Y', '5Y', '10Y']
swaption_data_conv = 'USD-SOFR-SWAPTION'
# nv
df = pd.DataFrame(np.random.uniform(low=90./1e4, high=110./1e4, size=(len(swaption_expiries), len(swaption_tenors))), index=swaption_expiries, columns=swaption_tenors)
data2D_nv_swpt = qfCreateData2D('Swaption Normal Volatility', swaption_data_conv, df)
# beta
df = pd.DataFrame(np.random.uniform(low=60./1e2, high=60./1e2, size=(len(swaption_expiries), len(swaption_tenors))), index=swaption_expiries, columns=swaption_tenors)
data2D_beta_swpt = qfCreateData2D('Swaption SABR Beta', swaption_data_conv, df)
# nu
df = pd.DataFrame(np.random.uniform(low=0.01, high=0.3, size=(len(swaption_expiries), len(swaption_tenors))), index=swaption_expiries, columns=swaption_tenors)
data2D_nu_swpt= qfCreateData2D('Swaption SABR Nu', swaption_data_conv, df)
# rho
df = pd.DataFrame(np.random.uniform(low=-0.99, high=0.99, size=(len(swaption_expiries), len(swaption_tenors))), index=swaption_expiries, columns=swaption_tenors)
data2D_rho_swpt = qfCreateData2D('Swaption SABR Rho', swaption_data_conv, df)

In [14]:
### capfloor
capfloor_expiries = ['1M', '3M', '6M', '1Y', '2Y']
capfloor_tenors = ['1M', '3M', '6M']
capfloor_data_conv = 'USD-SOFR-CAPFLOOR'
# nv
df = pd.DataFrame(np.random.uniform(low=90./1e4, high=110./1e4, size=(len(capfloor_expiries), len(capfloor_tenors))), index=capfloor_expiries, columns=capfloor_tenors)
data2D_nv_cf = qfCreateData2D('Swaption Normal Volatility', capfloor_data_conv, df)
# beta
df = pd.DataFrame(np.random.uniform(low=60./1e2, high=60./1e2, size=(len(capfloor_expiries), len(capfloor_tenors))), index=capfloor_expiries, columns=capfloor_tenors)
data2D_beta_cf = qfCreateData2D('Swaption SABR Beta', capfloor_data_conv, df)
# nu
df = pd.DataFrame(np.random.uniform(low=0.01, high=0.3, size=(len(capfloor_expiries), len(capfloor_tenors))), index=capfloor_expiries, columns=capfloor_tenors)
data2D_nu_cf= qfCreateData2D('Swaption SABR Nu', capfloor_data_conv, df)
# rho
df = pd.DataFrame(np.random.uniform(low=-0.99, high=0.99, size=(len(capfloor_expiries), len(capfloor_tenors))), index=capfloor_expiries, columns=capfloor_tenors)
data2D_rho_cf = qfCreateData2D('Swaption SABR Rho', capfloor_data_conv, df)

In [15]:
### create a data collection
data_collection_sabr_list = [
    data2D_nv_swpt, data2D_beta_swpt, data2D_nu_swpt, data2D_rho_swpt,
    data2D_nv_cf, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection_sabr = qfCreateDataCollection(data_collection_sabr_list)
data_collection_sabr.display()

,Data Shape,Data Type,Data Convention
0,DATA2D,Swaption Normal Volatility,USD-SOFR-SWAPTION
1,DATA2D,Swaption SABR Beta,USD-SOFR-SWAPTION
2,DATA2D,Swaption SABR Nu,USD-SOFR-SWAPTION
3,DATA2D,Swaption SABR Rho,USD-SOFR-SWAPTION
4,DATA2D,Swaption Normal Volatility,USD-SOFR-CAPFLOOR
5,DATA2D,Swaption SABR Beta,USD-SOFR-CAPFLOOR
6,DATA2D,Swaption SABR Nu,USD-SOFR-CAPFLOOR
7,DATA2D,Swaption SABR Rho,USD-SOFR-CAPFLOOR


### Create SABR model on top of the multi-indices YC model

In [16]:
sabr_model = qfCreateSABRModel(
    sub_model=yc_model,
    data_collection=data_collection_sabr,
    build_method_collection=bm_collection_sabr
)
print("SABR components:", sabr_model.components_.keys())

SABR components: dict_keys(['SOFR-1B-SWAPTION', 'SOFR-1B-CAPFLOOR'])


In [17]:
# Check interpolated SABR parameters for swaption and capfloor components
params_swpt = sabr_model.get_sabr_parameters(
    IndexRegistry().get("SOFR-1B-SWAPTION"),
    1.0,
    5.0,
)

params_cf = sabr_model.get_sabr_parameters(
    IndexRegistry().get("SOFR-1B-CAPFLOOR"),
    0.5,
    0.25,
)

print("Interpolated swaption SABR parameters at (1Y, 5Y):")
for k, v in params_swpt.items():
    print(k, v)

print("\nInterpolated capfloor SABR parameters at (6M, 3M):")
for k, v in params_cf.items():
    print(k, v)

Interpolated swaption SABR parameters at (1Y, 5Y):
SABRParameters.RHO -0.5780352251875405
SABRParameters.BETA 0.6
SABRParameters.NU 0.2144564782731271
SABRParameters.NV 0.009640797562990288

Interpolated capfloor SABR parameters at (6M, 3M):
SABRParameters.RHO -0.8804528193462265
SABRParameters.BETA 0.6
SABRParameters.NU 0.27560113680831416
SABRParameters.NV 0.009411958756214948


## SOFR Swaption Valuation

### Create SOFR swaption product

In [18]:
swaption = qfCreateProductRFRSwaption(
    expiry_date="2026-06-17",
    effective_date="2026-06-17",
    term_or_termination_date="5Y",
    payment_off_set="2D",
    on_index="SOFR-1B",
    strike=0.04,
    pay_or_rec="PAY",
    notional=1_000_000,
    accrual_period="1Y",
    accrual_basis="ACT/360",
    floating_leg_accrual_period="3M",
    pay_business_day_convention="F",
    pay_holiday_convention="USGS",
    spread=0.0,
    compounding_method="COMPOUND",
    long_or_short="LONG",
)
qfDisplayProduct(swaption)

,Name,Value
0,Product Type,PRODUCT_RFR_SWAPTION
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Expiry Date,2026-06-17
5,Effective Date,2026-06-17
6,Termination Date,2031-06-17
7,Payment Offset,2D
8,ON Index,SOFRON Actual/360
9,Strike,0.04


### Price the same swaption under two different funding curves

In [19]:
vp_funding_sofr = FundingIndexParameter({"Funding Index": "SOFR-1B-FLAT"})
vpc_sofr = ValuationParametersCollection([vp_funding_sofr])

vp_funding_ff = FundingIndexParameter({"Funding Index": "FF-1B-FLAT"})
vpc_ff = ValuationParametersCollection([vp_funding_ff])

request = ValuationRequest.PV

In [20]:
# price under SOFR funding
ve_swaption_sofr = ValuationEngineRFRSwaption(
    sabr_model,
    vpc_sofr,
    swaption,
    request,
)

ve_swaption_sofr.calculate_value()

print("Funding Curve    :", "SOFR-1B-FLAT")
print("PV               :", ve_swaption_sofr.value_)
print("Cash             :", ve_swaption_sofr.cash_)
print("Swap PV          :", ve_swaption_sofr.swap_value_)
print("Forward SwapRate :", ve_swaption_sofr.forward_swap_rate_)
print("Annuity          :", ve_swaption_sofr.annuity_)
print("Option Value     :", ve_swaption_sofr.option_value_)
print("Expiry Date      :", ve_swaption_sofr.expiry_date_)
print("Tenor            :", ve_swaption_sofr.tenor_)

Funding Curve    : SOFR-1B-FLAT
PV               : 2832.2815199579513
Cash             : 0.0
Swap PV          : -25529.204670000734
Forward SwapRate : 0.03437286718506216
Annuity          : 4.53680506744584
Option Value     : 0.0006242898863522226
Expiry Date      : June 17th, 2026
Tenor            : 5.072222222222222


In [21]:
# price under FF funding
ve_swaption_ff = ValuationEngineRFRSwaption(
    sabr_model,
    vpc_ff,
    swaption,
    request,
)

ve_swaption_ff.calculate_value()

print("Funding Curve    :", "FF-1B-FLAT")
print("PV               :", ve_swaption_ff.value_)
print("Cash             :", ve_swaption_ff.cash_)
print("Swap PV          :", ve_swaption_ff.swap_value_)
print("Forward SwapRate :", ve_swaption_ff.forward_swap_rate_)
print("Annuity          :", ve_swaption_ff.annuity_)
print("Option Value     :", ve_swaption_ff.option_value_)
print("Expiry Date      :", ve_swaption_ff.expiry_date_)
print("Tenor            :", ve_swaption_ff.tenor_)

Funding Curve    : FF-1B-FLAT
PV               : 2830.3972746997542
Cash             : 0.0
Swap PV          : -25474.76114605638
Forward SwapRate : 0.03437612791140685
Annuity          : 4.5297547214358955
Option Value     : 0.0006248455929204356
Expiry Date      : June 17th, 2026
Tenor            : 5.072222222222222


In [22]:
print(f"Swaption PV under SOFR funding : {ve_swaption_sofr.value_:.10f}")
print(f"Swaption PV under FF funding   : {ve_swaption_ff.value_:.10f}")
print(f"PV difference (FF - SOFR)      : {ve_swaption_ff.value_ - ve_swaption_sofr.value_:.10f}")

Swaption PV under SOFR funding : 2832.2815199580
Swaption PV under FF funding   : 2830.3972746998
PV difference (FF - SOFR)      : -1.8842452582


In [24]:
pv_report_sofr = qfCreateValueReport(sabr_model, swaption, vpc_sofr, "pv")
pv_report_ff = qfCreateValueReport(sabr_model, swaption, vpc_ff, "pv")

print("PV report under SOFR funding:")
display(pv_report_sofr)

print("PV report under FF funding:")
display(pv_report_ff)

PV report under SOFR funding:


[['USD', 2832.2815199579513]]

PV report under FF funding:


[['USD', 2830.3972746997542]]

In [26]:
cf_report_swaption_sofr = ve_swaption_sofr.create_cash_flows_report()
pv_cash_report_swaption_sofr = ve_swaption_sofr.get_value_and_cash()

display(cf_report_swaption_sofr.display())
display(pv_cash_report_swaption_sofr.display())

print("-----------------------------")

cf_report_swaption_ff = ve_swaption_ff.create_cash_flows_report()
pv_cash_report_swaption_ff = ve_swaption_ff.get_value_and_cash()

display(cf_report_swaption_ff.display())
display(pv_cash_report_swaption_ff.display())

,PRODUCT_TYPE,VALUATION_ENGINE_TYPE,LEG_ID,CASHFLOW_ID,PAY_OR_RECEIVE,NOTIONAL,PAY_DATE,FORECASTED_AMOUNT,PV,DISCOUNG FACTOR,FIXING_DATE,START_DATE,END_DATE,ACCRUED,INDEX_OR_FIXED,INDEX_VALUE
0,PRODUCT_RFR_SWAPTION,ValuationEngineRFRSwaption,0,0,1.0,1000000,"June 17th, 2026",2832.28152,2832.28152,1.0,"June 17th, 2026","June 17th, 2026","June 17th, 2031",4.536805,SOFRON Actual/360,0.034373


,Currency,Type,Value
0,USD,PV,2832.28152
1,USD,CASH,0.00000


-----------------------------


,PRODUCT_TYPE,VALUATION_ENGINE_TYPE,LEG_ID,CASHFLOW_ID,PAY_OR_RECEIVE,NOTIONAL,PAY_DATE,FORECASTED_AMOUNT,PV,DISCOUNG FACTOR,FIXING_DATE,START_DATE,END_DATE,ACCRUED,INDEX_OR_FIXED,INDEX_VALUE
0,PRODUCT_RFR_SWAPTION,ValuationEngineRFRSwaption,0,0,1.0,1000000,"June 17th, 2026",2830.397275,2830.397275,1.0,"June 17th, 2026","June 17th, 2026","June 17th, 2031",4.529755,SOFRON Actual/360,0.034376


,Currency,Type,Value
0,USD,PV,2830.397275
1,USD,CASH,0.000000


### Risk

In [28]:
grad = []
ve_swaption_sofr.calculate_first_order_risk(grad)

for i, g in enumerate(grad):
    print(f"component {i}: shape={g.shape}")
    print(g)

component 0: shape=(5,)
[-2036.44931099   -73.55273719  1024.91320588     0.
     0.        ]
component 1: shape=(5,)
[0. 0. 0. 0. 0.]
component 2: shape=(28,)
[  -489.29517948  44446.57677117  41011.23237114  47369.41779109
  44439.33134758  43133.66716066  43058.01033697  42962.20266771
  46252.5482745   41637.0290626   41622.17231077  41720.99180584
 147540.93623373 159007.4184047   53459.34728459      0.
      0.              0.              0.              0.
      0.              0.              0.              0.
      0.              0.              0.              0.        ]
component 3: shape=(5,)
[0. 0. 0. 0. 0.]
component 4: shape=(60,)
[0.00000000e+00 4.25989098e+05 6.24374656e+03 0.00000000e+00
 2.86574484e+05 4.20033860e+03 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 1.58780877e+02 2.32726039e+00 0.00000000e+00 1.06816226e+02
 1.56561154e+00 0.00000000e+00 0.0000

In [27]:
### test risk
df_risk = qfCreateValueReport(sabr_model, swaption, vpc_sofr, 'firstorderrisk').display()
df_risk

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,-2.564385e-01
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,5Y,,0.0,0.0001,1.408516e-01
2,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,10Y,,0.0,0.0001,2.462253e-01
3,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,20Y,,0.0,0.0001,4.633288e-22
4,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,30Y,,0.0,0.0001,0.000000e+00
...,...,...,...,...,...,...,...
158,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,1Y,3M,-0.528611,1.0,0.000000e+00
159,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,1Y,6M,0.594542,1.0,0.000000e+00
160,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,2Y,1M,0.777748,1.0,0.000000e+00
161,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,2Y,3M,-0.34429,1.0,0.000000e+00


In [29]:
df_risk[np.abs(df_risk["VALUES"].astype(float)) > 1e-10]

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,-0.256439
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,5Y,,0.0,0.0001,0.140852
2,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,10Y,,0.0,0.0001,0.246225
10,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,-6.396108
11,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,-0.081663
12,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-09-17x2026-12-10,,96.85,-0.01,-0.077104
13,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-12-10x2027-03-17,,96.9,-0.01,-0.030689
14,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2027-03-17x2027-06-16,,96.91,-0.01,0.071106
15,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2027-06-16x2027-09-15,,96.89,-0.01,-0.060245
16,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2027-09-15x2027-12-15,,96.85,-0.01,-0.067849


In [33]:
# SABR parameters
print("Normal Vol (NV)     :", ve_swaption_sofr.sabr_result_.get(SABRParameters.NV))
print("Beta                :", ve_swaption_sofr.sabr_result_.get(SABRParameters.BETA))
print("Nu                  :", ve_swaption_sofr.sabr_result_.get(SABRParameters.NU))
print("Rho                 :", ve_swaption_sofr.sabr_result_.get(SABRParameters.RHO))

# Pricing
print("Option Value        :", ve_swaption_sofr.option_value_)   # SABR/Bachelier price
print("Swap PV (underlying):", ve_swaption_sofr.swap_value_)    # intrinsic component
print("Total PV            :", ve_swaption_sofr.value_)


Normal Vol (NV)     : 0.010167250154706785
Beta                : 0.6
Nu                  : 0.1824875223034888
Rho                 : 0.41320328286986296
Option Value        : 0.0006242898863522226
Swap PV (underlying): -25529.204670000734
Total PV            : 2832.2815199579513


### Bump Reval for swaption

In [34]:
pv_base = qfCreateValueReport(sabr_model, swaption, vpc_sofr, "pv")[0][1]
print(f"Base PV: {pv_base}")

Base PV: 2832.2815199579513


In [35]:
risk_data_type = "SWAPTION NORMAL VOLATILITY"
risk_data_convention = "USD-SOFR-SWAPTION"
risk_expiry = "3M"
risk_tenor = "5Y"

bump_size = 1e-5

In [36]:
# Step 1: build bumped up model
df_nv_bumped = data2D_nv_swpt.display().copy()
cur_nv = df_nv_bumped.loc[risk_expiry, risk_tenor]
df_nv_bumped.loc[risk_expiry, risk_tenor] = cur_nv + bump_size
data2D_nv_bumped = qfCreateData2D(risk_data_type, risk_data_convention, df_nv_bumped)
data_list_bumped = [data2D_nv_bumped, data2D_beta_swpt, data2D_nu_swpt, data2D_rho_swpt,
                    data2D_nv_cf, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection_bumped = qfCreateDataCollection(data_list_bumped)
sabr_model_bumped = qfCreateSABRModel(sub_model=yc_model,
                                      data_collection= data_collection_bumped,
                                      build_method_collection=bm_collection_sabr)

# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(sabr_model_bumped, swaption, vpc_sofr, 'pv')[0][1]

# Step 3 : bump-reval risk
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.10f}.')

# Step 4: analytic risk
analytic_risk = df_risk[
    (df_risk["DATA_TYPE"] == risk_data_type)
    & (df_risk["DATA_CONVENTION"] == risk_data_convention)
    & (df_risk["AXIS1"] == risk_expiry)
    & (df_risk["AXIS2"] == risk_tenor)
]['VALUES'].values[0]
analytic_scaled = analytic_risk * bump_size
print(f'Analytic risk is {analytic_scaled}')
print(f"Difference is {risk_bump_reval - analytic_scaled:.10e}.")

Bump reval risk is 4.2608749927.
Analytic risk is 4.25989098002574
Difference is 9.8401272326e-04.
